# clipsmith — Local VOD (Anthropic / OpenAI)

Download a Twitch VOD and process it entirely inside Colab — no local machine required.

**What this notebook does:**
1. Installs clipsmith (CPU-only)
2. Downloads the VOD from Twitch with twitch-dl
3. Transcribes with faster-whisper on CPU (~25–40 min for a 2-hr VOD with `small`)
4. Selects the best moments with Anthropic Claude or OpenAI GPT
5. Cuts final clips with ffmpeg

**Requirements:**
- `TWITCH_CLIENT_ID` + `TWITCH_CLIENT_SECRET` in Colab Secrets (optional but recommended)
- `ANTHROPIC_API_KEY` **or** `OPENAI_API_KEY` in Colab Secrets

> For faster transcription, use [03_local_vod_ollama.ipynb](03_local_vod_ollama.ipynb) with a T4 GPU runtime.

In [ ]:
# @title 1. Install clipsmith
!pip install -q clipsmith-ai
!ffmpeg -version | head -1

In [ ]:
# @title 2. Load API keys from Colab Secrets
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") or ""
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
os.environ["TWITCH_CLIENT_ID"] = userdata.get("TWITCH_CLIENT_ID") or ""
os.environ["TWITCH_CLIENT_SECRET"] = userdata.get("TWITCH_CLIENT_SECRET") or ""

if not os.environ["ANTHROPIC_API_KEY"] and not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("Add ANTHROPIC_API_KEY or OPENAI_API_KEY to Colab Secrets")

provider = "anthropic" if os.environ["ANTHROPIC_API_KEY"] else "openai"
print(f"Using provider: {provider}")
if not os.environ["TWITCH_CLIENT_ID"]:
    print("Note: Twitch credentials not set — existing-clip boost signals will be skipped")

In [ ]:
# @title 3. Configure — set your VOD ID here
from pathlib import Path

VOD_ID = "2758856582"  # <-- paste your Twitch VOD ID
LANGUAGE = "es"  # ISO 639-1 — change to 'en' for English streams
WHISPER_MODEL = "small"  # tiny | small | medium | large-v3
MAX_CANDIDATES = 20

config_yaml = f"""\
work_dir: /content/work
out_dir:  /content/out

llm:
  provider: {provider}
  model_anthropic: claude-haiku-4-5-20251001
  model_openai: gpt-4.1-mini

transcribe:
  model: {WHISPER_MODEL}
  compute_type: int8
  language: {LANGUAGE}
  chunk_minutes: 10
  max_workers: 2

clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none

caption:
  enabled: false
"""

Path("/content/work").mkdir(parents=True, exist_ok=True)
Path("/content/out").mkdir(parents=True, exist_ok=True)
Path("config.yaml").write_text(config_yaml)
print(f"Config written — VOD: {VOD_ID}, LLM: {provider}, Whisper: {WHISPER_MODEL}")

In [ ]:
# @title 4. Run the full pipeline (download → transcribe → select → cut)
# Re-running this cell skips stages that already have cached output files.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "run-vod",
        VOD_ID,
        "--config",
        "config.yaml",
        "--max-candidates",
        str(MAX_CANDIDATES),
        "-v",
    ],
    check=True,
)

In [ ]:
# @title 5. Preview clips inline
from IPython.display import Video, display
from pathlib import Path

clips = sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4"))

if not clips:
    print("No clips found. Did step 4 complete successfully?")
else:
    print(f"Found {len(clips)} clip(s):")
    for mp4 in clips:
        print(f"  {mp4.name}")
        display(Video(str(mp4), embed=True, width=360))

In [ ]:
# @title 6. Download clips to your computer
from google.colab import files
from pathlib import Path

for mp4 in sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4")):
    files.download(str(mp4))